### Import libraries and load datasets

In [24]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = ROOT / "data" / "processed"
MODEL_DIR = ROOT / "models" / "bert" / "fine-tuned"
CHECKPOINT_DIR = ROOT / "models" / "bert" / "checkpoints"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "validation.csv")


### Prepare text and sentiment labels

In [25]:
TEXT_COLUMN = "clean_text"
LABEL_COLUMN = "sentiment"

for df in [train_df, val_df]:
    assert TEXT_COLUMN in df.columns, f"Missing column: {TEXT_COLUMN}"
    assert LABEL_COLUMN in df.columns, f"Missing column: {LABEL_COLUMN}"

def prepare_data(df):
    df = df[[TEXT_COLUMN, LABEL_COLUMN]].copy()

    df[TEXT_COLUMN] = df[TEXT_COLUMN].fillna("").astype(str).str.strip()
    df[LABEL_COLUMN] = (
        df[LABEL_COLUMN]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    df = df[
        (df[TEXT_COLUMN] != "") &
        (df[LABEL_COLUMN] != "")
    ].reset_index(drop=True)

    return df

train_df = prepare_data(train_df)
val_df = prepare_data(val_df)

print("Training rows:", len(train_df))
print("Validation rows:", len(val_df))
print("\nTraining labels:")
print(train_df[LABEL_COLUMN].value_counts())

Training rows: 15991
Validation rows: 1999

Training labels:
sentiment
positive    11475
negative     2686
neutral      1830
Name: count, dtype: int64


In [26]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Select the 'Python (.venv CUDA)' notebook kernel "
        "and restart the kernel."
    )

print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5050 Laptop GPU


### Encode sentiment labels

In [27]:
encoder = LabelEncoder()

train_df["label"] = encoder.fit_transform(train_df[LABEL_COLUMN])

known_labels = set(encoder.classes_)
validation_labels = set(val_df[LABEL_COLUMN].unique())

unknown_labels = validation_labels - known_labels

if unknown_labels:
    raise ValueError(f"Unknown validation labels: {unknown_labels}")

val_df["label"] = encoder.transform(val_df[LABEL_COLUMN])

label2id = {
    str(label): int(index)
    for index, label in enumerate(encoder.classes_)
}

id2label = {
    int(index): str(label)
    for index, label in enumerate(encoder.classes_)
}

print("Label mapping:", label2id)

Label mapping: {'negative': 0, 'neutral': 1, 'positive': 2}


### Load BERT tokenizer and model

In [28]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(encoder.classes_),
    label2id=label2id,
    id2label=id2label
)

print("BERT model loaded.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT model loaded.


### Create tokenized datasets

In [29]:
def make_dataset(df):
    dataset = Dataset.from_pandas(
        df[[TEXT_COLUMN, "label"]].rename(
            columns={TEXT_COLUMN: "text"}
        ),
        preserve_index=False
    )

    def tokenize(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=128
        )

    dataset = dataset.map(tokenize, batched=True)
    dataset = dataset.remove_columns(["text"])
    dataset = dataset.rename_column("label", "labels")

    return dataset


train_ds = make_dataset(train_df)
val_ds = make_dataset(val_df)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Tokenized training rows:", len(train_ds))
print("Tokenized validation rows:", len(val_ds))

Map:   0%|          | 0/15991 [00:00<?, ? examples/s]

Map:   0%|          | 0/1999 [00:00<?, ? examples/s]

Tokenized training rows: 15991
Tokenized validation rows: 1999


### Define evaluation metrics

In [30]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1
    }

### Configure training

In [31]:
training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=100,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=42
)

print("Training configuration ready.")

Training configuration ready.


### Train BERT

In [32]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1
1,0.426728,0.411662,0.844422,0.717776,0.680642,0.693974
2,0.319110,0.462110,0.850425,0.713913,0.689507,0.697118
3,0.243183,0.642308,0.843922,0.716958,0.698995,0.707203


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

### Evaluate on validation data

In [33]:
validation_results = trainer.evaluate()

print("Validation results:")

for metric, value in validation_results.items():
    if isinstance(value, (int, float)):
        print(f"{metric}: {value:.4f}")
    else:
        print(f"{metric}: {value}")

Training Loss,Validation Loss,Epoch,Accuracy,Macro Precision,Macro Recall,Macro F1
0.243183,0.642308,3,0.843922,0.716958,0.698995,0.707203


Validation results:
eval_loss: 0.6423
eval_accuracy: 0.8439
eval_macro_precision: 0.7170
eval_macro_recall: 0.6990
eval_macro_f1: 0.7072


### Save the best fine-tuned model

In [34]:
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))

joblib_path = MODEL_DIR / "label_encoder_classes.txt"

with open(joblib_path, "w", encoding="utf-8") as file:
    file.write("\n".join(map(str, encoder.classes_)))

print("BERT model saved to:", MODEL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

BERT model saved to: d:\My_Project\Analyse\mobile-review-sentiment-analysis\models\bert\fine-tuned
